# Coffee Rewards — Etapa 3: Preguntas de negocio (5–9)

Este notebook responde las preguntas de negocio de **temporalidad**, **gasto** y
**segmentación demográfica** (preguntas 5 a 9) usando **Pandas** sobre los datos
ya limpios de `data/processed/`. Las preguntas 1 a 4 (funnel y canales) se
responden con SQL en `sql/`.

Cada pregunta se documenta con el formato fijo:
**pregunta → método (SQL/Pandas) → hallazgo → insight**.

## Datos de entrada

- `customers_clean.csv` (17.000 × 6)
- `offers_clean.csv` (10 × 6)
- `events_clean.csv` (306.534 × 6)

Abajo se validan los conteos cruzados con las queries SQL (deben coincidir).

In [ ]:
"""Coffee Rewards — Etapa 3, preguntas 5-9 (Pandas).

Responde las preguntas de negocio de temporalidad, gasto y segmentación
demográfica sobre data/processed/ (datos ya limpios). No modifica data/raw/.
"""
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

customers = pd.read_csv("data/processed/customers_clean.csv")
offers = pd.read_csv("data/processed/offers_clean.csv")
events = pd.read_csv("data/processed/events_clean.csv")

# has_demographics viene como "True"/"False" -> coaccionar a bool
customers["has_demographics"] = (
    customers["has_demographics"].astype(str).str.lower().isin(["true", "1"])
)

# --- Validacion cruzada con SQL -------------------------------------------------
print("=" * 70)
print("VALIDACION CRUZADA (debe coincidir con SQL)")
print("=" * 70)
for ev, n in events["event"].value_counts().items():
    print(f"  {ev:<18} {n:>7}")
assert (events["event"] == "offer received").sum() == 76277
assert (events["event"] == "offer viewed").sum() == 57725
assert (events["event"] == "offer completed").sum() == 33579
print("  OK: received=76277 viewed=57725 completed=33579")

## Q5 — Tiempo entre recibir y ver/completar una oferta, por tipo

### Método
Pandas: agrupo los eventos por par (`customer_id`, `offer_id`), tomo la primera
hora de `offer received` como punto de partida y calculo la diferencia hasta la
primera `offer viewed` / `offer completed` posterior.

### Hallazgo
- **Tiempo hasta ver:** mediana 18 h (bogo 12 h, discount 18 h, informational 18 h).
- **Tiempo hasta completar:** mediana 54 h (bogo 42 h, discount 66 h); informational no se completa.

### Insight
El cliente decide rápido si abre la oferta (~18 h) pero tarda ~2–3 días en
alcanzar el umbral de gasto para redimirla. El **bogo** se completa más rápido
(42 h) que el **discount** (66 h), coherente con una recompensa más atractiva.

In [ ]:
# ==============================================================================
# Q5 — Tiempo entre recibir y ver/completar, por tipo de oferta
# ==============================================================================
rec = (
    events.loc[events["event"] == "offer received"]
    .groupby(["customer_id", "offer_id"])["time"]
    .min()
    .rename("rec_time")
)
view = (
    events.loc[events["event"] == "offer viewed"]
    .groupby(["customer_id", "offer_id"])["time"]
    .min()
    .rename("view_time")
)
comp = (
    events.loc[events["event"] == "offer completed"]
    .groupby(["customer_id", "offer_id"])["time"]
    .min()
    .rename("comp_time")
)
pairs = rec.to_frame().join(view, how="outer").join(comp, how="outer").reset_index()
pairs = pairs.merge(offers[["offer_id", "offer_type"]], on="offer_id")

# filtrar a eventos posteriores al recibido (evita pares anómalos)
pairs["time_to_view"] = np.where(pairs["view_time"] >= pairs["rec_time"],
                                 pairs["view_time"] - pairs["rec_time"], np.nan)
pairs["time_to_complete"] = np.where(pairs["comp_time"] >= pairs["rec_time"],
                                     pairs["comp_time"] - pairs["rec_time"], np.nan)

print("\n" + "=" * 70)
print("Q5 — TIEMPO HASTA VER (horas)")
print("=" * 70)
ttv = (
    pairs.dropna(subset=["time_to_view"])
    .groupby("offer_type")["time_to_view"]
    .agg(n="count", media_h="mean", mediana_h="median")
    .round(1)
)
print(ttv)
print("\nGlobal (tiempo hasta ver):")
g = pairs["time_to_view"].dropna()
print(f"  n={g.count()}  media={g.mean():.1f}h  mediana={g.median():.1f}h")

print("\n" + "=" * 70)
print("Q5 — TIEMPO HASTA COMPLETAR (horas)")
print("=" * 70)
ttc = (
    pairs.dropna(subset=["time_to_complete"])
    .groupby("offer_type")["time_to_complete"]
    .agg(n="count", media_h="mean", mediana_h="median")
    .round(1)
)
print(ttc)
print("\nGlobal (tiempo hasta completar):")
g = pairs["time_to_complete"].dropna()
print(f"  n={g.count()}  media={g.mean():.1f}h  mediana={g.median():.1f}h")

## Q6 — ¿Las ofertas incrementan el gasto?

### Método
Pandas: clasifico cada transacción según si en ese momento el cliente tenía una
oferta **activa** (recibida y dentro de `duration × 24` h) y, además, si la había
**visto**. Comparo el ticket medio entre grupos.

### Hallazgo
- sin oferta activa: 17.090 tx, media **12,76 USD**
- oferta activa no vista: 17.206 tx, media **12,18 USD**
- oferta activa vista: 104.657 tx, media **12,88 USD**

El 75% de las transacciones ocurren con una oferta vista activa.

### Insight
Las ofertas **no incrementan el ticket medio** (uplift ~ +0,12 USD, despreciable);
su valor está en **dirigir/concentrar el gasto** (redención), no en agrandar la
cesta. El baseline "sin oferta" es minoría (12%), así que la comparación es
descriptiva, no un control experimental limpio.

In [ ]:
# ==============================================================================
# Q6 — ¿Las ofertas incrementan el gasto?
# ==============================================================================
offers["dur_h"] = offers["duration"] * 24

trans = (
    events.loc[events["event"] == "transaction", ["customer_id", "time", "amount"]]
    .reset_index(drop=True)
)
trans["tx_id"] = trans.index

rec_e = (
    events.loc[events["event"] == "offer received", ["customer_id", "offer_id", "time"]]
    .rename(columns={"time": "rec_time"})
    .merge(offers[["offer_id", "dur_h"]], on="offer_id")
)
rec_e["expire"] = rec_e["rec_time"] + rec_e["dur_h"]

view_e = (
    events.loc[events["event"] == "offer viewed", ["customer_id", "offer_id", "time"]]
    .rename(columns={"time": "view_time"})
)

# transacciones con oferta activa (recibida y no expirada)
active = (
    trans.merge(rec_e[["customer_id", "offer_id", "rec_time", "expire"]],
                on="customer_id")
    .query("time >= rec_time and time <= expire")
)

# de esas, cuáles tenían además una oferta vista antes de la transacción
active_v = active.merge(view_e, on=["customer_id", "offer_id"])
active_v = active_v.query("view_time >= rec_time and view_time <= time")

tx_active = active["tx_id"].drop_duplicates()
tx_active_viewed = active_v["tx_id"].drop_duplicates()

trans["grupo"] = "sin_oferta_activa"
trans.loc[trans["tx_id"].isin(tx_active), "grupo"] = "oferta_activa_no_vista"
trans.loc[trans["tx_id"].isin(tx_active_viewed), "grupo"] = "oferta_activa_vista"

print("\n" + "=" * 70)
print("Q6 — GASTO MEDIO SEGUN CONTEXTO DE OFERTA")
print("=" * 70)
q6 = (
    trans.groupby("grupo")["amount"]
    .agg(n="count", media_usd="mean", total_usd="sum")
    .round(2)
    .reindex(["sin_oferta_activa", "oferta_activa_no_vista", "oferta_activa_vista"])
)
print(q6)

# uplift bruto descriptivo
m_no = q6.loc["sin_oferta_activa", "media_usd"]
m_vista = q6.loc["oferta_activa_vista", "media_usd"]
n_vista = q6.loc["oferta_activa_vista", "n"]
print(f"\n  uplift (vista vs sin oferta): {m_vista - m_no:+.2f} USD por transaccion")
print(f"  gasto incremental bruto (vista): {(m_vista - m_no) * n_vista:,.2f} USD")

## Q7 — Efectividad por segmento demográfico

### Método
Pandas: sobre la cohorte con demografía (`has_demographics=True`), calculo la
tasa de completado (completed/received) segmentando por edad, ingreso y género.

### Hallazgo
- **Edad:** crece de 38,1% (<30) hasta ~52% (50–70+).
- **Ingreso:** crece de 34,8% (<40k) hasta 62,3% (100k+).
- **Género:** F 56,4% · O 54,7% · M 43,2%.

### Insight
Los segmentos de **mayor edad e ingreso** redimen más; el de **menor ingreso
(<40k)** es el que menos. Las **mujeres** redimen ~13 p.p. más que los hombres.
Accionable: focalizar ofertas en ingreso alto/edad alta y rediseñar para M y <40k.

In [ ]:
# ==============================================================================
# Q7 — Efectividad por segmento demografico (solo cohorte con demografia)
# ==============================================================================
print("\n" + "=" * 70)
print("Q7 — RANGOS DE EDAD / INGRESO")
print("=" * 70)
demo = customers[customers["has_demographics"]]
print("  edad:", demo["age"].min(), "-", demo["age"].max(),
      "| ingreso:", demo["income"].min(), "-", demo["income"].max())

ev7 = (
    events[events["event"].isin(["offer received", "offer completed"])]
    .merge(customers[["customer_id", "gender", "age", "income", "has_demographics"]],
           on="customer_id")
)
ev7 = ev7[ev7["has_demographics"]]

ev7["age_bin"] = pd.cut(
    ev7["age"], bins=[0, 30, 40, 50, 60, 70, 200],
    labels=["<30", "30-39", "40-49", "50-59", "60-69", "70+"],
    right=False,
)
ev7["income_bin"] = pd.cut(
    ev7["income"], bins=[0, 40000, 60000, 80000, 100000, 200000],
    labels=["<40k", "40-59k", "60-79k", "80-99k", "100k+"],
    right=False,
)


def comp_rate(g):
    received = (g["event"] == "offer received").sum()
    completed = (g["event"] == "offer completed").sum()
    return pd.Series({
        "received": received,
        "completed": completed,
        "completion_rate_pct": round(100 * completed / received, 2),
    })


print("\nPor edad:")
print(ev7.groupby("age_bin").apply(comp_rate))
print("\nPor ingreso:")
print(ev7.groupby("income_bin").apply(comp_rate))
print("\nPor genero:")
print(ev7.groupby("gender").apply(comp_rate))

## Q8 — ¿La cohorte sin demografía se comporta distinto?

### Método
Pandas: comparo funnel (view/completion rate) y gasto entre las cohortes
`has_demographics=True` y `False`.

### Hallazgo
- **View rate:** sin-demo 80,4% vs con-demo 75,0%.
- **Completion rate:** sin-demo 11,6% vs con-demo 48,8%.
- **Ticket medio:** sin-demo 2,7 USD vs con-demo 14,0 USD.

### Insight
La cohorte sin demografía (2.175 clientes) se comporta como **cuentas anómalas**:
muchas transacciones muy chicas y casi no redimen. Conviene **excluirla** de los
análisis de efectividad por demo (por eso Q7 usa solo `has_demographics=True`) y
tratarla como posible artefacto/cuentas de prueba.

In [ ]:
# ==============================================================================
# Q8 — ¿La cohorte sin demografia se comporta distinto?
# ==============================================================================
print("\n" + "=" * 70)
print("Q8 — COHORTE CON vs SIN DEMOGRAFIA")
print("=" * 70)
ev8 = (
    events[events["event"].isin(["offer received", "offer viewed", "offer completed"])]
    .merge(customers[["customer_id", "has_demographics"]], on="customer_id")
)


def funnel_rates(g):
    received = (g["event"] == "offer received").sum()
    viewed = (g["event"] == "offer viewed").sum()
    completed = (g["event"] == "offer completed").sum()
    return pd.Series({
        "received": received,
        "viewed": viewed,
        "completed": completed,
        "view_rate_pct": round(100 * viewed / received, 2),
        "completion_rate_pct": round(100 * completed / received, 2),
    })


print(ev8.groupby("has_demographics").apply(funnel_rates))

trans8 = (
    events[events["event"] == "transaction"]
    .merge(customers[["customer_id", "has_demographics"]], on="customer_id")
)
print("\nGasto por cohorte:")
print(
    trans8.groupby("has_demographics")["amount"]
    .agg(n="count", media_usd="mean", total_usd="sum")
    .round(2)
)

## Q9 — Valor económico de las ofertas

### Método
Pandas: sumo las recompensas pagadas en los eventos `offer completed` y las
contrasto con el uplift de ticket estimado en Q6 (descriptivo, no causal).

### Hallazgo
- 33.579 ofertas completadas; recompensa total **164.676 USD** (media 4,90 USD).
- Por tipo: **bogo 113.440 USD** (69% del costo) · discount 51.236 · informational 0.
- Ticket medio global: 12,78 USD.

### Insight
El **bogo** concentra ~2/3 del costo en recompensas. Como el uplift de ticket es
~0 (Q6), el programa no agranda la cesta: los 164k USD son el precio por la
**redención/engagement**. Un ROI real requeriría análisis causal (fuera de
alcance).

In [ ]:
# ==============================================================================
# Q9 — Valor economico de las ofertas
# ==============================================================================
print("\n" + "=" * 70)
print("Q9 — VALOR ECONOMICO")
print("=" * 70)
completed = events[events["event"] == "offer completed"].merge(
    offers[["offer_id", "offer_type", "reward"]], on="offer_id", suffixes=("", "_off")
)
total_reward = completed["reward_off"].sum()
n_completed = len(completed)
total_tx = events.loc[events["event"] == "transaction", "amount"].sum()
n_tx = (events["event"] == "transaction").sum()

print(f"  ofertas completadas: {n_completed}")
print(f"  recompensa total pagada: {total_reward:,.0f} USD")
print(f"  recompensa media por completado: {total_reward / n_completed:.2f} USD")
print(f"  transacciones: {n_tx} | volumen total: {total_tx:,.2f} USD | "
      f"ticket medio: {total_tx / n_tx:.2f} USD")

print("\nRecompensa pagada por tipo de oferta:")
print(
    completed.groupby("offer_type")["reward_off"]
    .agg(n="count", total_usd="sum", media_usd="mean")
    .round(2)
)

# ROI bruto descriptivo (no causal)
print(f"\n  gasto incremental bruto (vista, Q6): {(m_vista - m_no) * n_vista:,.2f} USD")
print(f"  recompensa pagada: {total_reward:,.0f} USD")
print(f"  neto bruto (incremental - recompensa): "
      f"{(m_vista - m_no) * n_vista - total_reward:,.2f} USD (estimacion descriptiva)")